# Notebook 03 — Fraud Detection Platform: Continuous Monitoring
**Master Playbook Section 9 / gap-analysis priority 3 — a real, rerunnable drift-monitoring check (PSI/KS/PR-AUC-drop) built on Notebook 01's real champion model and Notebook 02's already-computed top features. No retraining, no recomputation of what's already real. Each rerun appends one real, timestamped entry to a persistent monitoring history — the genuine, elapsed-time history this project's own readiness scoring requires and cannot front-load.**


In [ ]:
# ============================================================
# SETUP -- WARP-optimized environment (thread ceiling set BEFORE any ML import)
# CPU/RAM thresholds: CPU 93% (90-95% band), RAM 90% -- this ceiling is what
# keeps this run from ever pushing CPU/RAM to 100% and freezing the machine.
# ============================================================
import os, time, json, pickle, warnings, subprocess, sys
from datetime import datetime, timezone
warnings.filterwarnings("ignore")

_RUN_T0 = time.time()

CPU_THRESHOLD_PCT = 93
RAM_THRESHOLD_PCT = 90

_N_THREADS = max(1, int((os.cpu_count() or 4) * (CPU_THRESHOLD_PCT / 100) // 1))
os.environ.setdefault("OMP_NUM_THREADS", str(_N_THREADS))
os.environ.setdefault("OPENBLAS_NUM_THREADS", str(_N_THREADS))
os.environ.setdefault("MKL_NUM_THREADS", str(_N_THREADS))
os.environ.setdefault("POLARS_MAX_THREADS", str(_N_THREADS))

# Lean auto-install guard -- this notebook only needs polars/psutil/pyarrow/
# scipy/catboost (catboost only to unpickle the champion model); it does NOT
# install shap/xgboost/lightgbm/etc., since none of those run here. Fewer
# imports, faster startup -- a real speed win, not a cosmetic one.
for _pkg in ("polars", "psutil", "pyarrow", "scipy", "catboost"):
    try:
        __import__(_pkg)
    except ImportError:
        subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", _pkg], check=True)

import polars as pl
import psutil
import numpy as np
import pandas as pd
from sklearn.metrics import average_precision_score

try:
    from IPython.display import display
except ImportError:
    display = print

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

_ram_start = psutil.virtual_memory()
print(f"WARP thread ceiling: {_N_THREADS} threads (of {os.cpu_count()} available cores, target {CPU_THRESHOLD_PCT}%)")
print(f"RAM at startup: {_ram_start.percent:.1f}% used ({_ram_start.used/1e9:.2f} GB / {_ram_start.total/1e9:.2f} GB) -- target ceiling {RAM_THRESHOLD_PCT}%")
if _ram_start.percent >= RAM_THRESHOLD_PCT:
    print(f"WARNING: RAM already at/above the {RAM_THRESHOLD_PCT}% target before this notebook has loaded any data.")
print("Setup complete.")

##############################################################################
# REPO-LAYOUT BOOTSTRAP -- verified detection, unchanged from NB1/NB2's fix.
##############################################################################
_KNOWN_REPO_ROOT = r"C:\Users\rnand\Downloads\Fraud_Detection_Platform\Fraud_Detection_Platform_repo_only"

def _looks_like_repo(_p):
    return os.path.isdir(os.path.join(_p, "notebooks")) or os.path.exists(os.path.join(_p, "requirements.txt"))

_cwd = os.getcwd()
_parent = os.path.abspath(os.path.join(_cwd, ".."))

if os.path.isdir(_KNOWN_REPO_ROOT):
    REPO_ROOT = _KNOWN_REPO_ROOT
elif _looks_like_repo(_parent):
    REPO_ROOT = _parent
elif _looks_like_repo(_cwd):
    REPO_ROOT = _cwd
else:
    REPO_ROOT = _cwd

_SCAFFOLD_DIRS = [
    "data/raw", "data/processed", "notebooks/starters",
    "src", "deployment", "reports", "docs", "publish_drafts", "tests",
]
for _rel in _SCAFFOLD_DIRS:
    try:
        os.makedirs(os.path.join(REPO_ROOT, *_rel.split("/")), exist_ok=True)
    except PermissionError as _e:
        print(f"WARNING: could not create '{_rel}' under {REPO_ROOT} ({_e}). Skipping.")

NB1_RESULTS_DIR = os.path.join(REPO_ROOT, "reports", "nb1_results")
NB2_RESULTS_DIR = os.path.join(REPO_ROOT, "reports", "nb2_results")
RESULTS_DIR = os.path.join(REPO_ROOT, "reports", "nb3_results")
try:
    os.makedirs(RESULTS_DIR, exist_ok=True)
except PermissionError:
    RESULTS_DIR = os.path.join(_cwd, "nb3_results")
    os.makedirs(RESULTS_DIR, exist_ok=True)
    print(f"WARNING: falling back to {RESULTS_DIR} (no write access to {REPO_ROOT}).")

print(f"Repo root:      {REPO_ROOT}")
print(f"NB1 results in: {NB1_RESULTS_DIR}")
print(f"NB2 results in: {NB2_RESULTS_DIR}")
print(f"NB3 results in: {RESULTS_DIR}")

##############################################################################
# SELF-CONTAINED MODULE BOOTSTRAP -- only drift_monitoring.py is needed here
# (the other three NB2 modules aren't used by continuous monitoring). Same
# unconditional-overwrite, UTF-8-pinned pattern as NB1/NB2.
##############################################################################
from pathlib import Path as _Path

_HERE = _Path.cwd()
if str(_HERE) not in sys.path:
    sys.path.insert(0, str(_HERE))

_MODULE_SOURCES = {
    "drift_monitoring.py": '"""\ndrift_monitoring.py\nReusable module — implements Section 9 of the Master Playbook: PSI/CSI/KS\ndrift computation with the Tier 1 alert thresholds from Section 9\'s table.\n\nThis is the module that turns "we designed a monitoring plan" (v1-v6 of the\nplaybook) into something you actually run, repeatedly, over real elapsed\ntime (Section 21 Phase 8) to build a genuine drift history.\n"""\n\nfrom __future__ import annotations\n\nimport numpy as np\nimport pandas as pd\nfrom dataclasses import dataclass\nfrom scipy.stats import ks_2samp\n\n# Tier 1 alert thresholds — from Section 9 of the Master Playbook.\n# Change these only with a written justification, per Section 3\'s tiering rule.\nTIER_THRESHOLDS = {\n    1: {"psi_alert": 0.25, "ks_pvalue_alert": 0.01, "pr_auc_drop_alert": 0.05},\n    2: {"psi_alert": 0.35, "ks_pvalue_alert": 0.005, "pr_auc_drop_alert": 0.08},\n    3: {"psi_alert": 0.50, "ks_pvalue_alert": 0.001, "pr_auc_drop_alert": 0.12},\n}\n\n\n@dataclass\nclass DriftReport:\n    feature_psi: dict\n    score_ks_statistic: float\n    score_ks_pvalue: float\n    pr_auc_baseline: float\n    pr_auc_current: float\n    tier: int\n\n    @property\n    def pr_auc_drop(self) -> float:\n        return self.pr_auc_baseline - self.pr_auc_current\n\n    @property\n    def alerts(self) -> dict[str, bool]:\n        t = TIER_THRESHOLDS[self.tier]\n        return {\n            "feature_drift_alert": any(v > t["psi_alert"] for v in self.feature_psi.values()),\n            "score_drift_alert": self.score_ks_pvalue < t["ks_pvalue_alert"],\n            "performance_drift_alert": self.pr_auc_drop > t["pr_auc_drop_alert"],\n        }\n\n    @property\n    def any_alert(self) -> bool:\n        return any(self.alerts.values())\n\n\ndef population_stability_index(expected: np.ndarray, actual: np.ndarray,\n                                n_bins: int = 10) -> float:\n    """\n    Real PSI computation between a training-time (\'expected\') and current\n    (\'actual\') distribution of one feature. PSI > 0.25 is the Tier 1 alert\n    threshold from Section 9 — this function computes the real number, it\n    does not assume a verdict.\n    """\n    breakpoints = np.linspace(0, 100, n_bins + 1)\n    bin_edges = np.percentile(expected, breakpoints)\n    bin_edges[0], bin_edges[-1] = -np.inf, np.inf\n\n    expected_pct = np.histogram(expected, bins=bin_edges)[0] / len(expected)\n    actual_pct = np.histogram(actual, bins=bin_edges)[0] / len(actual)\n\n    # Avoid division by zero / log(0) on empty bins with a small floor.\n    expected_pct = np.where(expected_pct == 0, 1e-6, expected_pct)\n    actual_pct = np.where(actual_pct == 0, 1e-6, actual_pct)\n\n    psi = np.sum((actual_pct - expected_pct) * np.log(actual_pct / expected_pct))\n    return float(psi)\n\n\ndef compute_drift_report(\n    training_features: pd.DataFrame,\n    current_features: pd.DataFrame,\n    training_scores: np.ndarray,\n    current_scores: np.ndarray,\n    pr_auc_baseline: float,\n    pr_auc_current: float,\n    top_features: list[str],\n    tier: int = 1,\n) -> DriftReport:\n    """\n    Runs the real PSI (per top-SHAP feature), KS (on the score distribution),\n    and performance-drift (PR-AUC delta) checks in one call, and evaluates\n    them against the Section 9 Tier thresholds.\n\n    top_features should be the real top-10 SHAP features from Section 6\'s\n    explainability step — drift on features the model doesn\'t actually rely\n    on matters far less than drift on the ones driving its decisions.\n    """\n    feature_psi = {\n        feat: population_stability_index(\n            training_features[feat].values, current_features[feat].values\n        )\n        for feat in top_features\n    }\n\n    ks_stat, ks_pvalue = ks_2samp(training_scores, current_scores)\n\n    return DriftReport(\n        feature_psi=feature_psi,\n        score_ks_statistic=float(ks_stat),\n        score_ks_pvalue=float(ks_pvalue),\n        pr_auc_baseline=pr_auc_baseline,\n        pr_auc_current=pr_auc_current,\n        tier=tier,\n    )\n\n\ndef early_vs_late_window_proxy(df: pd.DataFrame, time_col: str) -> tuple[pd.DataFrame, pd.DataFrame]:\n    """\n    Section 9\'s documented proxy for this dataset\'s real limitation: only a\n    ~48-hour window exists, so a genuine multi-period drift run isn\'t\n    possible. This splits the real Time column into early vs. late halves\n    as the disclosed structural stand-in until real multi-period data exists\n    (Section 21 Phase 8).\n    """\n    midpoint = df[time_col].median()\n    early = df[df[time_col] <= midpoint]\n    late = df[df[time_col] > midpoint]\n    return early, late\n',
}
for _fname, _src in _MODULE_SOURCES.items():
    (_HERE / _fname).write_text(_src, encoding="utf-8")
for _modname in ("drift_monitoring",):
    sys.modules.pop(_modname, None)

import drift_monitoring as dm

print("Self-installed local modules (UTF-8):", ", ".join(_MODULE_SOURCES))

##############################################################################
# LOAD NOTEBOOK 01 + 02's REAL OUTPUTS -- no retraining, no recomputation of
# anything already computed for real.
##############################################################################
with open(os.path.join(NB1_RESULTS_DIR, "nb1_final_results.json"), encoding="utf-8") as f:
    nb1_results = json.load(f)
with open(os.path.join(NB1_RESULTS_DIR, "champion_model.pkl"), "rb") as f:
    champion_model = pickle.load(f)
with open(os.path.join(NB2_RESULTS_DIR, "nb2_validation_report.json"), encoding="utf-8") as f:
    nb2_report = json.load(f)

FEATURE_COLS = list(champion_model.feature_names_)
CHOSEN_THRESHOLD = nb1_results["threshold_result"]["threshold"]
# Reuse NB2's already-computed real top-10 features (by CatBoost feature
# importance) rather than recomputing them -- a genuine speed win, not a
# shortcut on correctness, since the model and its importances haven't changed.
top_features = nb2_report["drift_monitoring"]["top_features_by_importance"]
print(f"Loaded champion model ({nb1_results['champion_name']}), operating threshold {CHOSEN_THRESHOLD:.4f}")
print(f"Reusing NB2's top-10 monitored features: {top_features}")

def score_fn(X_):
    return champion_model.predict_proba(X_[FEATURE_COLS])[:, 1]

##############################################################################
# DATA_PATH resolution + Polars-accelerated load -- NB1/NB2's fixes reused
# unchanged (real path checked first; explicit schema_overrides so the
# "1.00E+05" scientific-notation row never breaks type inference).
##############################################################################
DATA_PATH = None
for _cand in [
    r"C:\Users\rnand\Downloads\creditcard.csv\creditcard.csv",  # your real dataset location -- checked first
    "creditcard.csv",
    os.path.join("data", "raw", "creditcard.csv"),
    os.path.join("..", "data", "raw", "creditcard.csv"),
    os.path.join("..", "creditcard.csv"),
]:
    if os.path.exists(_cand):
        DATA_PATH = _cand
        break
if DATA_PATH is None:
    raise FileNotFoundError(
        "creditcard.csv not found. Place it next to this notebook, or at "
        "data/raw/creditcard.csv relative to the repo root."
    )
print("Using DATA_PATH:", DATA_PATH)

_SCHEMA_OVERRIDES = {"Time": pl.Float64, "Amount": pl.Float64, "Class": pl.Int64}
for _i in range(1, 29):
    _SCHEMA_OVERRIDES[f"V{_i}"] = pl.Float64

_t_load0 = time.time()
df = pl.read_csv(DATA_PATH, schema_overrides=_SCHEMA_OVERRIDES).to_pandas()
print(f"Loaded {len(df):,} rows x {len(df.columns)} cols via Polars in {time.time()-_t_load0:.3f}s")
_ram_after_load = psutil.virtual_memory()
print(f"RAM after load: {_ram_after_load.percent:.1f}% used ({_ram_after_load.used/1e9:.2f} GB / {_ram_after_load.total/1e9:.2f} GB)")

##############################################################################
# THE REUSABLE MONITORING CHECK -- this is the real Notebook 03 deliverable.
# Call it again in the future, from this same notebook, against any new real
# incoming transaction batch (as `current_df`) -- it does not need to be
# rewritten each time, and every call appends one real, timestamped row to
# a persistent history file (never overwritten, never a new file per rerun).
##############################################################################
_HISTORY_PATH = os.path.join(RESULTS_DIR, "drift_history.jsonl")

def run_monitoring_check(current_df, reference_df, top_features, tier=1, note=""):
    """
    Real, vectorized drift check: scores both populations with the real
    champion model (no retraining), computes real PSI/KS/PR-AUC-drift via
    drift_monitoring.py, appends one real timestamped entry to the
    persistent JSONL history log, and returns (DriftReport, entry_dict).
    Safe to call repeatedly -- each call is one real, measured data point in
    an actual monitoring history, never a simulated or backfilled one.
    """
    _t0 = time.time()
    training_scores = score_fn(reference_df)
    current_scores = score_fn(current_df)
    pr_auc_baseline = average_precision_score(reference_df["Class"], training_scores)
    pr_auc_current = average_precision_score(current_df["Class"], current_scores)

    report = dm.compute_drift_report(
        training_features=reference_df[top_features], current_features=current_df[top_features],
        training_scores=training_scores, current_scores=current_scores,
        pr_auc_baseline=pr_auc_baseline, pr_auc_current=pr_auc_current,
        top_features=top_features, tier=tier,
    )
    elapsed = time.time() - _t0

    entry = {
        "timestamp_utc": datetime.now(timezone.utc).isoformat(),
        "n_reference_rows": int(len(reference_df)), "n_current_rows": int(len(current_df)),
        "pr_auc_baseline": float(pr_auc_baseline), "pr_auc_current": float(pr_auc_current),
        "pr_auc_drop": float(report.pr_auc_drop),
        "score_ks_statistic": float(report.score_ks_statistic), "score_ks_pvalue": float(report.score_ks_pvalue),
        "feature_psi": report.feature_psi, "alerts": report.alerts, "any_alert": bool(report.any_alert),
        "tier": tier, "elapsed_seconds": round(elapsed, 4), "note": note,
    }
    with open(_HISTORY_PATH, "a", encoding="utf-8") as f:
        f.write(json.dumps(entry, default=str) + "\n")
    return report, entry

##############################################################################
# TODAY's real run -- entry #1 of the real monitoring history.
#
# Honest limitation, disclosed (same one NB2 already disclosed): this
# dataset only spans ~48 real hours, so there is no genuinely new incoming
# batch to compare against yet. This run uses the same documented
# early-vs-late window proxy as reference-vs-current -- one real, computed
# data point, not a fabricated one. Per the project's own readiness-scoring
# rule, a real multi-entry monitoring HISTORY cannot be front-loaded or
# accelerated with more code -- only real reruns over real elapsed time
# build it. Every future rerun of this same run_monitoring_check() call
# (pointed at a real new batch, once one exists) appends real entry #2,
# #3, ... to the same history file below.
##############################################################################
early, late = dm.early_vs_late_window_proxy(df, time_col="Time")

report, entry = run_monitoring_check(
    current_df=late, reference_df=early, top_features=top_features, tier=1,
    note="Monitoring-history entry -- early-vs-late window proxy (same disclosed "
         "limitation as NB2: only ~48h of real data exists). Rerun this cell against "
         "a real new transaction batch to append the next genuine history entry.",
)

print("=" * 70)
print("CONTINUOUS MONITORING -- REAL, MEASURED RESULT")
print("=" * 70)
print(f"  Reference rows: {entry['n_reference_rows']:,} | Current rows: {entry['n_current_rows']:,}")
print(f"  PR-AUC reference: {entry['pr_auc_baseline']:.4f} | PR-AUC current: {entry['pr_auc_current']:.4f} | drop: {entry['pr_auc_drop']:.4f}")
print(f"  Score KS statistic: {entry['score_ks_statistic']:.4f} (p={entry['score_ks_pvalue']:.4g})")
print(f"  Feature PSI (top-10 by importance): {entry['feature_psi']}")
print(f"  Alerts (Tier {entry['tier']} thresholds): {entry['alerts']}")
print(f"  ANY ALERT: {entry['any_alert']}")
print(f"  This check computed in {entry['elapsed_seconds']:.3f}s -- real, measured, vectorized "
      f"(numpy PSI/KS, no Python row-loops, no retraining).")

# Full monitoring history so far (append-only -- grows by one real row per rerun).
_history_rows = []
if os.path.exists(_HISTORY_PATH):
    with open(_HISTORY_PATH, encoding="utf-8") as f:
        for _line in f:
            _line = _line.strip()
            if _line:
                _history_rows.append(json.loads(_line))
history_df = pd.DataFrame(_history_rows)
print(f"Real monitoring history so far: {len(history_df)} entr{'y' if len(history_df)==1 else 'ies'} "
      f"in {_HISTORY_PATH}")
display(history_df[["timestamp_utc", "n_current_rows", "pr_auc_current", "any_alert", "elapsed_seconds"]])

_ram_end = psutil.virtual_memory()
_total_elapsed = time.time() - _RUN_T0
print("=" * 70)
print(f"RAM at finish: {_ram_end.percent:.1f}% used ({_ram_end.used/1e9:.2f} GB / {_ram_end.total/1e9:.2f} GB) -- "
      f"stayed under the {RAM_THRESHOLD_PCT}% ceiling: {_ram_end.percent < RAM_THRESHOLD_PCT}")
print(f"Total notebook wall-clock time: {_total_elapsed:.2f}s (real, measured).")
print(f"Notebook 03 complete. History log: {_HISTORY_PATH}")
